# **Задание 2.** Трансформеры

## Импорт библиотек и модулей

In [1]:
%%capture
!pip install polars transformers datasets accelerate evaluate peft bitsandbytes

In [2]:
import os
import random

import torch
import transformers as tfm
import datasets as dts
import polars as pl
import pandas as pd
import evaluate
import peft

import sklearn as sl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings
warnings.filterwarnings("ignore")

Зададим значение `seed` и зафиксируем его:

In [3]:
seed_value = 927

random.seed(seed_value)
torch.manual_seed(seed_value)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.mps.is_available() else
    "cpu"
)
print(f"Using device: {device}")

Using device: mps


## Подготовка датасета, токенизация

In [5]:
from datasets import load_dataset
import polars as pl

# Загружаем датасет правильно
dataset = load_dataset("MonoHime/ru_sentiment_dataset")

# Конвертируем в Polars DataFrame
train_df = pl.from_arrow(dataset["train"].data.table)
val_df = pl.from_arrow(dataset["validation"].data.table)

In [6]:
train_df.head()

Unnamed: 0,text,sentiment
i64,str,i64
21098,""".с.,и спросил его: о Посланни…",1
21099,"""Роднее всех родных Попала я в …",1
21100,"""Непорядочное отношение к своим…",2
21101,"""). Отсутствуют нормативы, Гост…",1
21102,""" У меня машина в рук…",1


In [7]:
MODEL_NAME = "cointegrated/rubert-tiny-sentiment-balanced"

In [8]:
sentiment = tfm.pipeline("sentiment-analysis", model=MODEL_NAME)

Device set to use mps:0


In [9]:
tokenizer = tfm.AutoTokenizer.from_pretrained(MODEL_NAME)
model = tfm.AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)

In [10]:
def tokenize(batch):
    return tokenizer(batch, padding="max_length", truncation=True, max_length=128)

In [11]:
def apply_tokenization(df: pl.dataframe):
    _tokenization = tokenize(df["text"].to_list())
    df_tok = df.with_columns([
        pl.Series("input_ids", _tokenization["input_ids"]),
        pl.Series("attention_mask", _tokenization["attention_mask"])
    ])
    df_tok = df_tok.drop("text")
    return df_tok

In [12]:
train_df_tok = apply_tokenization(train_df)
val_df_tok = apply_tokenization(val_df)

In [13]:
train_df_tok.head(5)

Unnamed: 0,sentiment,input_ids,attention_mask
i64,i64,list[i64],list[i64]
21098,1,"[2, 18, … 3]","[1, 1, … 1]"
21099,1,"[2, 16915, … 3]","[1, 1, … 1]"
21100,2,"[2, 6226, … 0]","[1, 1, … 0]"
21101,1,"[2, 13, … 3]","[1, 1, … 1]"
21102,1,"[2, 299, … 3]","[1, 1, … 1]"


In [14]:
# NOTE: optinal for faster training on data sample (N samples per class)
N_SAMPLES = 20000
train_df_tok_sample = train_df_tok.group_by("sentiment").map_groups(
    lambda x: x.sample(
        n=min(N_SAMPLES, len(x)),
        with_replacement=False,
        shuffle=True,
        seed=seed_value
    )
)

In [15]:
class SentimentDataset(torch.utils.data.Dataset):
    """
    Custom PyTorch Dataset for polars DataFrame.
    """
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Создание экземпляра датасета
train_dataset = SentimentDataset(
    input_ids=torch.tensor(train_df_tok['input_ids'].to_list()),
    attention_mask=torch.tensor(train_df_tok['attention_mask'].to_list()),
    labels=torch.tensor(train_df_tok['sentiment'].to_list())
)
val_dataset = SentimentDataset(
    input_ids=torch.tensor(val_df_tok['input_ids'].to_list()),
    attention_mask=torch.tensor(val_df_tok['attention_mask'].to_list()),
    labels=torch.tensor(val_df_tok['sentiment'].to_list())
)

## Обучение и валидация модели

In [16]:
args = tfm.TrainingArguments(
    output_dir="./output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=2e-5,
    load_best_model_at_end=True,
    fp16=(device == torch.device("cuda")),
    report_to="none"
)

In [17]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [18]:
trainer = tfm.Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.546800,0.526594,0.752962,0.742865,0.735042,0.736757
2,0.486600,0.509831,0.766613,0.761441,0.753594,0.757157
3,0.445600,0.509379,0.770642,0.760180,0.762865,0.761478
4,0.408600,0.519850,0.771732,0.759058,0.768328,0.763163
5,0.363900,0.539044,0.771732,0.760735,0.764732,0.762648
6,0.307600,0.597042,0.768604,0.758944,0.754514,0.756356


KeyboardInterrupt: 

In [20]:
metrics = trainer.evaluate()
metrics

{'eval_loss': 0.5970415472984314,
 'eval_accuracy': 0.768603659114608,
 'eval_precision': 0.7589436537589019,
 'eval_recall': 0.7545136454999364,
 'eval_f1': 0.7563560258883815}

## Визуализация результатов

In [34]:
val_dataset_sample = SentimentDataset(
    input_ids=val_dataset.input_ids[:6],
    attention_mask=val_dataset.attention_mask[:6],
    labels=val_dataset.labels[:6]
)

In [35]:
samples = val_dataset_sample
preds = trainer.predict(samples)
labels = np.argmax(preds.predictions, axis=-1)

for text, true_label, pred_label in zip(val_df["text"], preds.label_ids, labels):
    print(f"Текст: {text}")
    print(f"Реальная метка: {true_label}, Предсказанная: {pred_label}\n")

Текст: Развода на деньги нет Наблюдаюсь в Лайфклиник по беременности, развода на деньги нет, врачи не плохие, по поводу ресепшен соглашусь (путают документы). 
Реальная метка: 1, Предсказанная: 2

Текст: Отель выбрали потому что рядом со стадионом. Отель 4*. Номер большой. Кровать 2-спальная одна. 2 одеяла. Много подушек. Есть зона отдыха. Чайный сет. 2 бутылки по 0,5 л бесплатно каждый день. Много шкафов. Мини-бар. Утюг и гладильная доска, отдельно прибор для глажки брюк. Санузел общий большой. Ванна, лейка съемная. 2 раковины, фен. Халаты, тапочки. Расширенный пакет косметических средств, даже соль для ванны. Интернет быстрый. Так как этаж высокий был, вид из окна на город. Один недостаток в номере 919: он напротив хозлифта и с утра начинает пользоваться им персонал для обслуживания завтрака на 9 этаже. Очень слышно звон посуды и разговоры. Завтрак не включен, стоит 23 евро. Рядом с отелем есть кафе и subway.
Реальная метка: 0, Предсказанная: 0

Текст: Вылечили Гноился с рождения гла

## Вывод

Обучение было остановлено, так как после стабильного роста метрик был резкий скачок. Возможно, что дальше бы обучение стабилизировалось, если это локальный всплеск, но скорее всего не в рамках заданных 10 эпох, поэтому, можно сказать, вручную смоделировал early stopping. 

Предсказания получились не очень точными, но предсказуемыми. К примеру, последний отзыв начинается с позитивных высказываний, поэтому модель могла принять его за позитивный, хотя в нем рассматриваются и минусы. В целом, метрики на валидации получились достаточно высокими.